### Playwright를 사용한 크롤링

Playwright는 브라우저를 코드로 제어하는 라이브러리이다. 페이지를 열고 JavaScript가 반영된 DOM에서 데이터를 읽을 수 있다.

```text
브라우저 실행 → 페이지 이동 → 제목 요소 대기 → 3초간 화면 확인 → 제목 수집 → 브라우저 종료
```

### 라이브러리 준비

터미널에서 Playwright 패키지와 Chromium 브라우저를 설치한다.

```bash
pip install playwright
python -m playwright install chromium
```

### 실습 파일 실행

실행 코드는 [09-playwright-crawling.py](09-playwright-crawling.py)에 작성되어 있다. 프로젝트 최상위 폴더에서 터미널을 열고 실행한다.

```bash
python html_crwaling/09-playwright-crawling.py
```

브라우저가 열리고 제목이 나타나면 화면을 3초 동안 보여준다. 이후 터미널에 웹툰 제목을 출력하고 종료한다. 아래에서는 이 파일의 코드를 부분별로 살펴본다.

### 브라우저 실행

Chromium 브라우저를 실행하고 `new_page()`로 탭을 만든다.

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=False)
    page = browser.new_page()
```

- `headless=False`: 브라우저 창을 표시한다.
- `headless=True`: 창을 표시하지 않는다. 화면이 없는 서버 환경에서 사용할 수 있다.

### 페이지 이동

`page.goto()`로 웹툰 페이지에 접속한다. 

```python
URL = "https://comic.naver.com/webtoon?tab=mon"
response = page.goto(URL, wait_until="domcontentloaded")
```

`domcontentloaded`는 초기 HTML 문서가 준비될 때까지 기다린다. JavaScript가 나중에 표시하는 제목은 별도로 기다려야 한다.

### Locator로 제목 찾기

**Locator**는 페이지에서 특정 요소를 찾는 객체이다. `page.locator()`에 CSS 선택자를 전달한다.

개발자 도구에서 웹툰 제목을 검사하고 클래스 이름을 확인한다. 이 예제는 `.ContentTitle__title--e3qXt`로 제목을 찾는다.

```python
TITLE_SELECTOR = ".ContentTitle__title--e3qXt"
title_locator = page.locator(TITLE_SELECTOR)
```

사이트 구조가 바뀌면 현재 DOM에 맞게 선택자를 수정한다.

### 제목 요소 기다리기

**페이지가 열렸다고 해서 수집할 데이터까지 모두 준비된 것은 아니다.** 초기 HTML을 받은 뒤 JavaScript가 서버에서 데이터를 받아 화면에 추가할 수 있다. 이때 페이지 이동이 끝나도 웹툰 제목은 아직 표시되지 않았을 수 있다.

제목이 나타나기 전에 목록을 읽으면 빈 결과를 얻을 수 있으므로, **수집할 요소가 나타날 때까지 기다린 뒤 데이터를 읽어야 한다.**

`page.locator()`는 어떤 요소를 찾을지 지정한다. Locator를 만드는 것만으로 요소가 나타날 때까지 기다리지는 않는다. 아래 코드는 `first.wait_for()`로 첫 번째 제목이 보이는 상태를 기다린다.

```python
title_locator.first.wait_for(state="visible", timeout=15000)
```

- `state="visible"`: 요소가 보이는 상태를 기다린다.
- `timeout=15000`: 최대 15초 동안 기다린다. 제목이 먼저 나타나면 바로 다음 코드로 진행하고, 시간 안에 나타나지 않으면 오류가 발생한다.

제목이 표시된 뒤에는 화면을 살펴볼 수 있도록 3초간 더 기다린다. 이 대기는 수업 중 화면을 확인하기 위한 시간이다. `wait_for_timeout()`의 시간 단위는 밀리초이다.

```python
page.wait_for_timeout(3000)
```

### 제목 수집

`all()`로 현재 선택자에 맞는 Locator 목록을 가져오고, `text_content()`로 각 요소의 텍스트를 읽는다.

```python
for title in title_locator.all():
    text = title.text_content()
    if text and text.strip():
        print(text.strip())
```

`all()`은 요소가 나타날 때까지 기다리지 않는다. 앞에서 첫 제목이 나타난 것을 확인한 뒤 현재 DOM의 제목들을 읽는다. 추가로 로딩되는 목록은 별도의 대기 조건이 필요하다.

### 브라우저 종료

`browser.close()`로 브라우저를 닫는다. 실행 파일에서는 수집 코드를 `try`에 두고 `finally`에서 브라우저를 닫아 오류가 나도 정리한다.

```python
browser.close()
```

`with` 블록이 끝나면 Playwright도 종료된다.